In [2]:
"""
Section V.A — Group 1 TODOs
============================

TODO 1: Replace fixed message state with random message sampling.
        Draw N_MSG random states uniformly on the Bloch sphere,
        run the full Method-I perturbation sweep for each,
        then plot mean ± std of all-qubit fidelity curves.

TODO 2: Replace "statevector confirmation" with a quantitative
        finite-shot tomography fidelity table:
        run post-selected circuit with shot noise, reconstruct ρ_Y
        via Pauli tomography, compute F(ρ_Y, ρ_M) with bootstrap CIs.

Both use the deterministic decoder from adding_unitaries.py (Method I:
R_X(θ) applied to E *before* scrambling).
"""

import numpy as np
import warnings
warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from math import pi
from collections import defaultdict
from scipy.stats import beta as beta_dist

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import (Statevector, DensityMatrix,
                                  partial_trace, state_fidelity)
from qiskit_aer import AerSimulator

# =============================================================================
#  SHARED PARAMETERS
# =============================================================================
N_MSG        = 20      # number of random message states
N_THETA      = 50      # points in the perturbation sweep
SHOTS        = 20000   # shots per tomography circuit
N_BOOTSTRAP  = 2000    # bootstrap resamples for CI
SEED         = 42
RNG          = np.random.default_rng(SEED)

THETA_FIXED  = 2.5349076035276403   # original fixed message
VARPHI_FIXED = 2.0022404587009195

QUBIT_LABELS = {0:"C (CTC)", 1:"E", 2:"R", 3:"G", 4:"M", 5:"A", 6:"Y"}
OUT          = '/Users/nandan/Desktop/CTCs/IBM/figures_sec_5'

# =============================================================================
#  CIRCUIT BUILDERS (from adding_unitaries.py, Method I)
# =============================================================================
def build_decoder_method1(theta_rx, theta_msg, varphi_msg):
    """
    Method I: R_X(theta_rx) applied to E *before* scrambling.
    Returns 7-qubit statevector circuit (no measurements).
    Register order: C=0, E=1, R=2, G=3, M=4, A=5, Y=6
    """
    C=QuantumRegister(1,'C'); E=QuantumRegister(1,'E'); R=QuantumRegister(1,'R')
    A=QuantumRegister(1,'A'); M=QuantumRegister(1,'M'); G=QuantumRegister(1,'G')
    Y=QuantumRegister(1,'Y')
    qc=QuantumCircuit(C,E,R,G,M,A,Y)

    qc.u(theta_msg, varphi_msg, 0.0, M[0])
    qc.swap(C[0], M[0]); qc.barrier()

    qc.h(E[0]); qc.cx(E[0], M[0])
    qc.h(R[0]); qc.cx(R[0], G[0])
    qc.h(A[0]); qc.cx(A[0], Y[0]); qc.barrier()

    # Method I perturbation: R_X(theta_rx) on E BEFORE scrambling
    qc.rx(theta_rx, E[0]); qc.barrier()

    # Scrambling unitary
    qc.cz(C[0],R[0]); qc.cz(E[0],R[0]); qc.cz(C[0],E[0])
    qc.h(C[0]); qc.h(E[0]); qc.h(R[0])
    qc.cz(C[0],R[0]); qc.cz(C[0],E[0]); qc.cz(E[0],R[0]); qc.barrier()

    # Decoder U†
    qc.cz(A[0],G[0]); qc.cz(M[0],A[0]); qc.cz(G[0],M[0])
    qc.h(A[0]); qc.h(M[0]); qc.h(G[0])
    qc.cz(A[0],G[0]); qc.cz(G[0],M[0]); qc.cz(M[0],A[0]); qc.barrier()

    # Grover (R,G)
    qc.rz(pi,R[0]); qc.rx(pi,R[0]); qc.rx(pi,G[0])
    qc.swap(R[0],G[0]); qc.rz(pi,R[0]); qc.barrier()

    # Unitary Transpose
    qc.cz(A[0],G[0]); qc.cz(M[0],A[0]); qc.cz(G[0],M[0])
    qc.h(A[0]); qc.h(M[0]); qc.h(G[0])
    qc.cz(A[0],G[0]); qc.cz(G[0],M[0]); qc.cz(M[0],A[0]); qc.barrier()

    # Grover (A,Y)
    qc.rz(pi,A[0]); qc.rx(pi,A[0]); qc.rx(pi,Y[0])
    qc.swap(A[0],Y[0]); qc.rz(pi,A[0]); qc.barrier()

    # Unitary Conjugate again
    qc.cz(A[0],G[0]); qc.cz(M[0],A[0]); qc.cz(G[0],M[0])
    qc.h(A[0]); qc.h(M[0]); qc.h(G[0])
    qc.cz(A[0],G[0]); qc.cz(G[0],M[0]); qc.cz(M[0],A[0]); qc.barrier()

    # Grover (R,G) again
    qc.rz(pi,R[0]); qc.rx(pi,R[0]); qc.rx(pi,G[0])
    qc.swap(R[0],G[0]); qc.rz(pi,R[0]); qc.barrier()

    # Bell projection (unitary part)
    qc.cx(R[0],G[0]); qc.h(R[0])
    return qc


def postselect_RG_00(psi):
    """Project onto R=0, G=0; return (psi_post, p_succ)."""
    data = psi.data.copy()
    new  = np.zeros_like(data, dtype=complex)
    for i in range(len(data)):
        bits = format(i,'07b')   # C E R G M A Y
        if bits[2]=='0' and bits[3]=='0':
            new[i] = data[i]
    norm = np.linalg.norm(new)
    if norm < 1e-12:
        return None, 0.0
    return Statevector(new/norm), float(norm**2)


def fidelity_qubit_q(q, theta_rx, theta_msg, varphi_msg):
    """Statevector fidelity of qubit q after post-selection."""
    qc  = build_decoder_method1(theta_rx, theta_msg, varphi_msg)
    psi = Statevector.from_instruction(qc)
    psi_post, p = postselect_RG_00(psi)
    if psi_post is None:
        return 0.0, 0.0
    rho_q = partial_trace(DensityMatrix(psi_post),
                          [i for i in range(7) if i != q])
    qcm   = QuantumCircuit(1); qcm.u(theta_msg, varphi_msg, 0.0, 0)
    psi_M = Statevector.from_instruction(qcm)
    return float(state_fidelity(rho_q, psi_M)), p

# =============================================================================
#  HELPER: random message states uniformly on Bloch sphere
# =============================================================================
def random_message_params(n, rng):
    """
    Draw n random (theta_msg, varphi_msg) pairs corresponding to
    states sampled uniformly from the Bloch sphere.
    theta in [0, pi], varphi in [0, 2*pi].
    """
    # Uniform on sphere: cos(theta) ~ Uniform(-1,1)
    cos_theta = rng.uniform(-1.0, 1.0, n)
    theta     = np.arccos(cos_theta)          # in [0, pi]
    varphi    = rng.uniform(0.0, 2*pi, n)     # in [0, 2*pi]
    return list(zip(theta, varphi))


# =============================================================================
#  TODO 1 — RANDOM MESSAGE SAMPLING
#  For each of N_MSG random message states, sweep theta_rx across [0, pi]
#  and record fidelity of every qubit. Then plot mean ± std.
# =============================================================================
print("="*60)
print("  TODO 1: Random message state sampling")
print("="*60)

thetas_rx = np.linspace(0, pi, N_THETA)
msg_params = random_message_params(N_MSG, RNG)

# Add the fixed message as the first entry so we can compare
msg_params = [(THETA_FIXED, VARPHI_FIXED)] + list(msg_params)
N_TOTAL    = len(msg_params)   # N_MSG + 1

print(f"  {N_TOTAL} message states ({N_MSG} random + 1 fixed)")
print(f"  {N_THETA} theta_rx values per state")
print(f"  Total circuits: {N_TOTAL * N_THETA * 7}\n")

# fidelities_all[msg_idx, qubit, theta_idx]
fidelities_all = np.zeros((N_TOTAL, 7, N_THETA))
probs_all      = np.zeros((N_TOTAL, N_THETA))

for m, (th_msg, vph_msg) in enumerate(msg_params):
    label = "fixed" if m == 0 else f"rand{m}"
    print(f"  Message {m+1}/{N_TOTAL}  (θ={th_msg:.3f}, φ={vph_msg:.3f})", end=" ... ")
    for ti, th_rx in enumerate(thetas_rx):
        for q in range(7):
            F, p = fidelity_qubit_q(q, th_rx, th_msg, vph_msg)
            fidelities_all[m, q, ti] = F
        probs_all[m, ti] = p
    print("done")

# Statistics across random messages (exclude fixed message at index 0)
F_rand   = fidelities_all[1:, :, :]   # shape (N_MSG, 7, N_THETA)
F_mean   = F_rand.mean(axis=0)        # (7, N_THETA)
F_std    = F_rand.std(axis=0)         # (7, N_THETA)
F_fixed  = fidelities_all[0, :, :]   # (7, N_THETA) — original fixed message

p_mean   = probs_all[1:].mean(axis=0)
p_std    = probs_all[1:].std(axis=0)
p_fixed  = probs_all[0]

print(f"\n  Sampling complete.")
print(f"  Mean F(Y) at θ_rx=0:    {F_mean[6,0]:.4f} ± {F_std[6,0]:.4f}")
print(f"  Mean F(Y) at θ_rx=π/2:  {F_mean[6,N_THETA//2]:.4f} ± {F_std[6,N_THETA//2]:.4f}")
print(f"  Mean F(Y) at θ_rx=π:    {F_mean[6,-1]:.4f} ± {F_std[6,-1]:.4f}")

# ── Colour palette matching original paper figures ────────────────────────────
COLORS = {0:'#1f77b4',1:'#ff7f0e',2:'#2ca02c',3:'#d62728',
          4:'#9467bd',5:'#8c564b',6:'#e377c2'}

# ── Figure 1a: Mean ± std for all qubits ─────────────────────────────────────
fig1a, ax = plt.subplots(figsize=(8, 5))

for q in range(7):
    c = COLORS[q]
    ax.plot(thetas_rx, F_mean[q], color=c, lw=1.8,
            label=QUBIT_LABELS[q])
    ax.fill_between(thetas_rx,
                    F_mean[q] - F_std[q],
                    F_mean[q] + F_std[q],
                    color=c, alpha=0.18)

# Overlay fixed-message result as dashed
ax.plot(thetas_rx, F_fixed[6], color=COLORS[6], lw=1.2, ls='--',
        alpha=0.7, label='Y  (fixed msg)')

ax.set_xlabel(r'$\theta$ in $R_x(\theta)$ on $E$', fontsize=13)
ax.set_ylabel('Fidelity with input message $\\rho_M$', fontsize=13)
ax.set_title('Method I: fidelity of each qubit vs $\\theta$\n'
             f'Mean $\\pm 1\\sigma$ over {N_MSG} random message states',
             fontsize=13)
ax.set_ylim(-0.05, 1.08)
ax.legend(fontsize=10, ncol=2, framealpha=0.9)
ax.grid(alpha=0.2)
ax.tick_params(labelsize=11)
fig1a.tight_layout()
fig1a.savefig(OUT+'fig_VA_method1_random_sampling.pdf', dpi=200,
              bbox_inches='tight')
plt.close(fig1a)
print("\n  [✓] fig_VA_method1_random_sampling.pdf")

# ── Figure 1b: Output qubit Y only — fixed vs mean±std ───────────────────────
fig1b, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)

for ax, (data_mean, data_std, data_fixed, title, ylabel) in zip(axes, [
    (F_mean[6], F_std[6], F_fixed[6],
     'Output qubit $Y$: fidelity vs $\\theta$',
     'Fidelity $F(\\rho_Y, \\rho_M)$'),
    (p_mean, p_std, p_fixed,
     'Post-selection $p_{\\rm succ}$ vs $\\theta$',
     '$p_{\\rm succ} = P(R=0, G=0)$'),
]):
    ax.plot(thetas_rx, data_mean, color='#2980B9', lw=2.0,
            label=f'Mean over {N_MSG} random msgs')
    ax.fill_between(thetas_rx,
                    data_mean - data_std,
                    data_mean + data_std,
                    color='#2980B9', alpha=0.22,
                    label='$\\pm 1\\sigma$')
    ax.plot(thetas_rx, data_fixed, color='#C0392B', lw=1.5, ls='--',
            label='Fixed msg ($\\theta_m=2.53$)')
    ax.set_xlabel(r'$\theta$ in $R_x(\theta)$ on $E$', fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=12)
    ax.set_ylim(-0.05, 1.08)
    ax.legend(fontsize=9.5, framealpha=0.9)
    ax.grid(alpha=0.2)
    ax.tick_params(labelsize=11)

fig1b.suptitle('Section V.A — Method I perturbation: random message sampling',
               fontsize=12, y=1.01)
fig1b.tight_layout()
fig1b.savefig(OUT+'fig_VA_method1_Y_and_psucc.pdf', dpi=200,
              bbox_inches='tight')
plt.close(fig1b)
print("  [✓] fig_VA_method1_Y_and_psucc.pdf")


# =============================================================================
#  TODO 2 — FINITE-SHOT TOMOGRAPHY FIDELITY TABLE
#  Run the post-selected decoder with shot noise (AerSimulator).
#  For each of N_MSG_TOMO message states, reconstruct ρ_Y via Pauli
#  tomography, compute F(ρ_Y, ρ_M) with bootstrap 95% CI.
#  Report as a LaTeX-ready table and a figure.
# =============================================================================
print("\n" + "="*60)
print("  TODO 2: Finite-shot tomography fidelity table")
print("="*60)

N_MSG_TOMO   = 8    # messages for the table (compact)
SHOTS_TOMO   = SHOTS

# Build tomography circuits (post-selected, Pauli bases Z/X/Y on output qubit Y)
def build_tomo_shot_circuit(basis, theta_msg, varphi_msg):
    """
    Post-selected probabilistic decoder (simple version, no Grover) with
    Pauli rotation on the output register C before measurement.
    Uses the same circuit structure as the hardware pipeline.
    """
    C=QuantumRegister(1,'C'); E=QuantumRegister(1,'E'); R=QuantumRegister(1,'R')
    G=QuantumRegister(1,'G'); M=QuantumRegister(1,'M'); A=QuantumRegister(1,'A')
    Yr=QuantumRegister(1,'Y')
    crR=ClassicalRegister(1,'crR'); crG=ClassicalRegister(1,'crG')
    crT=ClassicalRegister(1,'crTomo')
    qc=QuantumCircuit(C,E,R,G,M,A,Yr,crR,crG,crT)

    qc.u(theta_msg,varphi_msg,0.0,M); qc.swap(C,M); qc.barrier()
    qc.h(E); qc.cx(E,M); qc.h(R); qc.cx(R,G); qc.h(A); qc.cx(A,Yr); qc.barrier()
    qc.cz(C,R); qc.cz(E,R); qc.cz(C,E); qc.h(C); qc.h(E); qc.h(R)
    qc.cz(C,R); qc.cz(C,E); qc.cz(E,R); qc.barrier()
    qc.cz(A,G); qc.cz(M,A); qc.cz(G,M); qc.h(A); qc.h(M); qc.h(G)
    qc.cz(A,G); qc.cz(G,M); qc.cz(M,A); qc.barrier()
    qc.cx(R,G); qc.h(R); qc.barrier()
    qc.swap(C,Yr)
    qc.measure(R,crR); qc.measure(G,crG)
    if basis=='X': qc.h(C)
    elif basis=='Y': qc.sdg(C); qc.h(C)
    qc.measure(C,crT)
    return qc


def parse_bs(bs):
    p = bs.split(' ') if ' ' in bs else list(bs)
    return p[0], p[1], p[2]   # crTomo, crG, crR


def postselect_counts(counts):
    kept=defaultdict(int); nt=0; nk=0
    for bs,cnt in counts.items():
        nt+=cnt
        crT,crG,crR=parse_bs(bs)
        if crR=='0' and crG=='0':
            kept[crT]+=cnt; nk+=cnt
    return dict(kept), nt, nk


def pauli_exp(c):
    n0=c.get('0',0); n1=c.get('1',0); N=n0+n1
    return (n0-n1)/N if N>0 else 0.0


def reconstruct_dm(sx,sy,sz):
    X=np.array([[0,1],[1,0]],dtype=complex)
    Y=np.array([[0,-1j],[1j,0]],dtype=complex)
    Z=np.array([[1,0],[0,-1]],dtype=complex)
    rho=(np.eye(2)+sx*X+sy*Y+sz*Z)/2
    ev,evec=np.linalg.eigh(rho); ev=np.maximum(ev,0); ev/=ev.sum()
    return (evec*ev)@evec.conj().T


def cp_ci(k, n, alpha=0.05):
    lo=beta_dist.ppf(alpha/2,k,n-k+1) if k>0 else 0.0
    hi=beta_dist.ppf(1-alpha/2,k+1,n-k) if k<n else 1.0
    return float(lo), float(hi)


def bootstrap_fidelity(tX, tY, tZ, rho_M, n=N_BOOTSTRAP):
    bsF=np.zeros(n)
    for i in range(n):
        def rs(d):
            keys=list(d.keys()); vals=np.array([d[k] for k in keys])
            N=vals.sum()
            if N==0: return {'0':1,'1':1}
            new=RNG.multinomial(N,vals/N); return {k:int(v) for k,v in zip(keys,new)}
        sx=pauli_exp(rs(tX)); sy=pauli_exp(rs(tY)); sz=pauli_exp(rs(tZ))
        bsF[i]=state_fidelity(DensityMatrix(reconstruct_dm(sx,sy,sz)), rho_M)
    return bsF


sim = AerSimulator()

# Message states: fixed + N_MSG_TOMO-1 random ones from our earlier sample
tomo_msgs = [(THETA_FIXED, VARPHI_FIXED)] + msg_params[1:N_MSG_TOMO]

print(f"  Running tomography for {len(tomo_msgs)} message states × {SHOTS_TOMO:,} shots\n")

tomo_results = []

for m, (th_msg, vph_msg) in enumerate(tomo_msgs):
    label = "Fixed" if m==0 else f"Rand {m}"
    # ideal F from statevector
    qcm=QuantumCircuit(1); qcm.u(th_msg,vph_msg,0.0,0)
    rho_M = DensityMatrix(Statevector.from_instruction(qcm))

    tomo={}; nt_z=0; nk_z=0
    for basis in ['Z','X','Y']:
        qc=build_tomo_shot_circuit(basis,th_msg,vph_msg)
        from qiskit import transpile
        t=transpile(qc,sim,optimization_level=1,seed_transpiler=SEED)
        res=sim.run(t,shots=SHOTS_TOMO).result()
        kept,nt,nk=postselect_counts(res.get_counts(t))
        tomo[basis]=kept
        if basis=='Z': nt_z,nk_z=nt,nk

    sx=pauli_exp(tomo['X']); sy=pauli_exp(tomo['Y']); sz=pauli_exp(tomo['Z'])
    rho_Y=reconstruct_dm(sx,sy,sz)
    F_shot=float(state_fidelity(DensityMatrix(rho_Y),rho_M))

    bsF=bootstrap_fidelity(tomo['X'],tomo['Y'],tomo['Z'],rho_M)
    F_lo=float(np.percentile(bsF,2.5)); F_hi=float(np.percentile(bsF,97.5))

    # Ideal statevector fidelity for comparison
    qc_sv=build_decoder_method1(0.0,th_msg,vph_msg)   # theta_rx=0 = unperturbed
    psi_sv=Statevector.from_instruction(qc_sv)
    psi_post,p_sv=postselect_RG_00(psi_sv)
    if psi_post is not None:
        rho_Y_sv=partial_trace(DensityMatrix(psi_post),[0,1,2,3,4,5])
        F_sv=float(state_fidelity(rho_Y_sv,rho_M))
    else:
        F_sv=float('nan')

    p_hat=nk_z/nt_z
    cp_lo,cp_hi=cp_ci(nk_z,nt_z)

    tomo_results.append({
        'label'   : label,
        'th_msg'  : th_msg,
        'vph_msg' : vph_msg,
        'F_shot'  : F_shot,
        'F_lo'    : F_lo,
        'F_hi'    : F_hi,
        'F_sv'    : F_sv,
        'p_succ'  : p_hat,
        'p_lo'    : cp_lo,
        'p_hi'    : cp_hi,
        'n_kept'  : nk_z,
        'n_total' : nt_z,
        'sx'      : sx, 'sy': sy, 'sz': sz,
        'bootstrap_F': bsF.tolist(),
    })
    print(f"  {label:8s}  θ={th_msg:.3f}  φ={vph_msg:.3f}  "
          f"F={F_shot:.4f} [{F_lo:.4f},{F_hi:.4f}]  "
          f"F_sv={F_sv:.4f}  p={p_hat:.4f}  nkept={nk_z}")


# ── Figure 2: Fidelity comparison — finite-shot vs statevector ───────────────
fig2, axes2 = plt.subplots(1, 2, figsize=(11, 4.5))

msg_labels = [r['label'] for r in tomo_results]
F_vals  = [r['F_shot'] for r in tomo_results]
F_elos  = [max(0.0, r['F_shot'] - r['F_lo']) for r in tomo_results]
F_ehis  = [max(0.0, r['F_hi']  - r['F_shot']) for r in tomo_results]
F_sv    = [r['F_sv']   for r in tomo_results]
p_vals  = [r['p_succ'] for r in tomo_results]
p_elos  = [max(0.0, r["p_succ"] - r["p_lo"]) for r in tomo_results]
p_ehis  = [max(0.0, r["p_hi"] - r["p_succ"]) for r in tomo_results]

x = np.arange(len(tomo_results))

# (a) Fidelity F(ρ_Y, ρ_M)
ax = axes2[0]
ax.bar(x, F_vals, color='#2980B9', width=0.5,
       yerr=[F_elos, F_ehis],
       error_kw=dict(elinewidth=1.5, capsize=5, capthick=1.5, ecolor='#222'),
       zorder=3, edgecolor='white',
       label=f'Finite-shot tomo ({SHOTS_TOMO:,} shots/basis)')
ax.scatter(x, F_sv, color='#27AE60', s=60, zorder=5, marker='D',
           label='Statevector (ideal)')
ax.axhline(1.0, color='#27AE60', ls='--', lw=1.0, alpha=0.5)
ax.set_xticks(x); ax.set_xticklabels(msg_labels, fontsize=9.5)
ax.set_ylim(0.5, 1.08)
ax.set_ylabel('$F(\\rho_Y, \\rho_M)$', fontsize=12)
ax.set_title('(a) Output fidelity per message state\n'
             '(bars = finite-shot tomo ± 95% bootstrap CI;\n'
             ' diamonds = statevector ideal)', fontsize=11)
ax.yaxis.grid(True, alpha=0.3, zorder=0)
ax.legend(fontsize=9, framealpha=0.9)
for xi, fv in zip(x, F_vals):
    ax.text(xi, fv+F_ehis[xi]+0.008, f'{fv:.4f}', ha='center', fontsize=8)

# (b) p_succ
ax2 = axes2[1]
ax2.bar(x, p_vals, color='#8E44AD', width=0.5,
        yerr=[p_elos, p_ehis],
        error_kw=dict(elinewidth=1.5, capsize=5, capthick=1.5, ecolor='#222'),
        zorder=3, edgecolor='white')
ax2.axhline(0.25, color='#27AE60', ls='--', lw=1.2, alpha=0.7,
            label='Ideal $p_{\\rm succ}=0.25$')
ax2.set_xticks(x); ax2.set_xticklabels(msg_labels, fontsize=9.5)
ax2.set_ylim(0, 0.35)
ax2.set_ylabel('$p_{\\rm succ}$', fontsize=12)
ax2.set_title('(b) Post-selection rate per message state\n'
              f'(95% Clopper–Pearson CI, {SHOTS_TOMO:,} shots/basis)',
              fontsize=11)
ax2.yaxis.grid(True, alpha=0.3, zorder=0)
ax2.legend(fontsize=9.5, framealpha=0.9)
for xi, pv in zip(x, p_vals):
    ax2.text(xi, pv+p_ehis[xi]+0.006, f'{pv:.4f}', ha='center', fontsize=8)

fig2.suptitle('Section V.A — Finite-shot tomography: $F_{\\rm msg}$ and $p_{\\rm succ}$\n'
              f'(noiseless AerSimulator, {len(tomo_results)} message states,'
              f' $N_{{\\rm bootstrap}}={N_BOOTSTRAP}$)',
              fontsize=12, y=1.02)
fig2.tight_layout()
fig2.savefig(OUT+'fig_VA_tomography_fidelity.pdf', dpi=200, bbox_inches='tight')
plt.close(fig2)
print("\n  [✓] fig_VA_tomography_fidelity.pdf")


# ── LaTeX-ready table (paper-ready) ──────────────────────────────────────────
print("\n" + "="*60)
print("  PAPER-READY TABLE (LaTeX source)")
print("="*60)

latex = r"""\begin{table}[h]
\centering
\caption{Finite-shot tomography fidelities for representative message states.
$F_{\rm sv}$ is the noiseless statevector reference; $F_{\rm tomo}$ is
reconstructed from """ + f"{SHOTS_TOMO:,}" + r""" shots per Pauli basis with
95\% bootstrap confidence intervals; $p_{\rm succ}$ has 95\% Clopper--Pearson CI.
All results use the noiseless AerSimulator.}
\label{tab:tomo_fidelities}
\begin{tabular}{lcccccc}
\hline
Message & $\theta_m$ & $\varphi_m$ & $F_{\rm sv}$ & $F_{\rm tomo}$ & 95\% CI & $p_{\rm succ}$ \\
\hline
"""

for r in tomo_results:
    latex += (
        f"{r['label']:8s} & "
        f"{r['th_msg']:.4f} & "
        f"{r['vph_msg']:.4f} & "
        f"{r['F_sv']:.4f} & "
        f"{r['F_shot']:.4f} & "
        f"$[{r['F_lo']:.4f},\\,{r['F_hi']:.4f}]$ & "
        f"{r['p_succ']:.4f} \\\\\n"
    )

latex += r"""\hline
\end{tabular}
\end{table}"""

print(latex)

# Also save as plain text for quick reference
print("\n\n  Plain text table:")
print(f"  {'Msg':8s}  {'θ_m':>7}  {'φ_m':>7}  {'F_sv':>7}  "
      f"{'F_tomo':>7}  {'95% CI':>20}  {'p_succ':>7}  {'N_kept':>7}")
print("  " + "-"*75)
for r in tomo_results:
    print(f"  {r['label']:8s}  {r['th_msg']:>7.4f}  {r['vph_msg']:>7.4f}  "
          f"{r['F_sv']:>7.4f}  {r['F_shot']:>7.4f}  "
          f"[{r['F_lo']:.4f},{r['F_hi']:.4f}]  "
          f"{r['p_succ']:>7.4f}  {r['n_kept']:>7d}")

print(f"\n  [✓] All Group 1 outputs saved to {OUT}")
print("  Figures: fig_VA_method1_random_sampling.pdf")
print("           fig_VA_method1_Y_and_psucc.pdf")
print("           fig_VA_tomography_fidelity.pdf")

  TODO 1: Random message state sampling
  21 message states (20 random + 1 fixed)
  50 theta_rx values per state
  Total circuits: 7350

  Message 1/21  (θ=2.535, φ=2.002) ... done
  Message 2/21  (θ=0.991, φ=4.763) ... done
  Message 3/21  (θ=1.693, φ=2.228) ... done
  Message 4/21  (θ=0.771, φ=6.099) ... done
  Message 5/21  (θ=1.165, φ=5.612) ... done
  Message 6/21  (θ=2.518, φ=4.891) ... done
  Message 7/21  (θ=0.314, φ=1.223) ... done
  Message 8/21  (θ=1.021, φ=2.932) ... done
  Message 9/21  (θ=0.962, φ=0.275) ... done
  Message 10/21  (θ=2.409, φ=0.969) ... done
  Message 11/21  (θ=1.670, φ=4.292) ... done
  Message 12/21  (θ=1.832, φ=4.679) ... done
  Message 13/21  (θ=0.548, φ=6.079) ... done
  Message 14/21  (θ=1.279, φ=2.047) ... done
  Message 15/21  (θ=0.869, φ=2.328) ... done
  Message 16/21  (θ=1.684, φ=2.950) ... done
  Message 17/21  (θ=2.148, φ=1.190) ... done
  Message 18/21  (θ=1.461, φ=0.816) ... done
  Message 19/21  (θ=2.631, φ=2.989) ... done
  Message 20/21  